In [1]:
 !pip install -q monai nibabel
import os, glob, tarfile, zipfile, json, shutil, random, time
import numpy as np, pandas as pd, torch, torch.nn.functional as F, nibabel as nib
from pathlib import Path
from scipy.stats import spearmanr
from monai import transforms
from monai.data import Dataset, DataLoader
from monai.losses import DiceLoss
from monai.inferers import sliding_window_inference
from monai.metrics import DiceMetric
from monai.networks.nets import SwinUNETR

START = time.time()
TRAIN_LIMIT_H = 6.5
random.seed(0); np.random.seed(0); torch.manual_seed(0)
device = torch.device("cuda")

# ===== 1. Unpack: 0-248 train/val, 250-349 test =====
TAR = glob.glob("/kaggle/input/**/BraTS2021_Training_Data.tar", recursive=True)[0]
OUT = "/kaggle/working/data"
os.makedirs(OUT, exist_ok=True)
order = {}
with tarfile.open(TAR, "r|") as t:
    for m in t:
        if not m.isfile():
            continue
        case = os.path.basename(os.path.dirname(m.name))
        if case not in order:
            order[case] = len(order)
            if order[case] >= 350:
                break
        if order[case] < 249 or 250 <= order[case] < 350:
            t.extract(m, OUT, filter="data")
MODS = ["flair", "t1ce", "t1", "t2"]
trainval, test_files = [], []
for d in sorted(Path(OUT).glob("**/BraTS2021_*")):
    if not d.is_dir() or d.name not in order:
        continue
    imgs = [str(d / f"{d.name}_{m}.nii.gz") for m in MODS]
    seg = d / f"{d.name}_seg.nii.gz"
    if not (all(os.path.exists(p) for p in imgs) and seg.exists()):
        continue
    item = {"image": imgs, "label": str(seg), "id": d.name}
    (trainval if order[d.name] < 249 else test_files).append(item)
random.shuffle(trainval)
train_files, val_files = trainval[:199], trainval[199:249]
print(len(train_files), "train |", len(val_files), "val |", len(test_files), "test", flush=True)

# ===== 2. Train NORMAL model =====
train_tf = transforms.Compose([
    transforms.LoadImaged(keys=["image", "label"], ensure_channel_first=True),
    transforms.ConvertToMultiChannelBasedOnBratsClassesd(keys="label"),
    transforms.CropForegroundd(keys=["image", "label"], source_key="image", k_divisible=[96, 96, 96]),
    transforms.RandSpatialCropd(keys=["image", "label"], roi_size=[96, 96, 96], random_size=False),
    transforms.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    transforms.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
    transforms.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
    transforms.NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    transforms.RandScaleIntensityd(keys="image", factors=0.1, prob=1.0),
    transforms.RandShiftIntensityd(keys="image", offsets=0.1, prob=1.0),
])
tf = transforms.Compose([
    transforms.LoadImaged(keys=["image", "label"], ensure_channel_first=True),
    transforms.ConvertToMultiChannelBasedOnBratsClassesd(keys="label"),
    transforms.NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
])
train_loader = DataLoader(Dataset(train_files, train_tf), batch_size=1, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(Dataset(val_files, tf), batch_size=1, num_workers=4)

net = SwinUNETR(in_channels=4, out_channels=3, feature_size=48).to(device)
loss_fn = DiceLoss(smooth_nr=0, smooth_dr=1e-5, squared_pred=True, sigmoid=True)
EPOCHS, VAL_EVERY = 100, 2
opt = torch.optim.AdamW(net.parameters(), lr=1e-4, weight_decay=1e-5)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler()
vmetric = DiceMetric(include_background=True, reduction="mean_batch")
history = {"epoch": [], "train_loss": [], "val_epoch": [], "val_dice": [], "val_tc": [], "val_wt": [], "val_et": []}
best = -1
NORMAL_PTH = "/kaggle/working/normal_model.pth"

for ep in range(1, EPOCHS + 1):
    t0 = time.time(); net.train(); total = 0
    for b in train_loader:
        x, y = b["image"].to(device), b["label"].float().to(device)
        opt.zero_grad(set_to_none=True)
        with torch.autocast("cuda"):
            loss = loss_fn(net(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        total += loss.item()
    sched.step()
    tl = total / len(train_loader)
    history["epoch"].append(ep); history["train_loss"].append(tl)
    msg = f"Epoch {ep}/{EPOCHS}  loss {tl:.4f}  ({(time.time()-t0)/60:.1f} min)"
    if ep % VAL_EVERY == 0 or ep == EPOCHS:
        net.eval(); vmetric.reset()
        with torch.no_grad():
            for b in val_loader:
                x, y = b["image"].to(device), b["label"].float().to(device)
                with torch.autocast("cuda"):
                    o = sliding_window_inference(x, (96, 96, 96), 4, net, overlap=0.5)
                vmetric(y_pred=(torch.sigmoid(o.float()) > 0.5).float(), y=y)
        tc, wt, et = vmetric.aggregate().tolist(); vd = (tc + wt + et) / 3
        for k, v in zip(["val_epoch", "val_dice", "val_tc", "val_wt", "val_et"], [ep, vd, tc, wt, et]):
            history[k].append(v)
        msg += f" | val Dice {vd:.4f} (TC {tc:.3f} WT {wt:.3f} ET {et:.3f})"
        if vd > best:
            best = vd
            torch.save(net.state_dict(), NORMAL_PTH)
            msg += "  -> saved new best model"
    print(msg, flush=True)
    json.dump(history, open("/kaggle/working/history.json", "w"))
    if (time.time() - START) / 3600 > TRAIN_LIMIT_H:
        print(f"Time limit reached at epoch {ep}, moving on to the test.", flush=True)
        break
print(f"TRAINING DONE. Best val Dice: {best:.4f}", flush=True)
del net, opt, scaler; torch.cuda.empty_cache()

# ===== 3. Load models =====
def load(path):
    m = SwinUNETR(in_channels=4, out_channels=3, feature_size=48)
    sd = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(sd, dict) and "state_dict" in sd:
        sd = sd["state_dict"]
    m.load_state_dict(sd)
    return m.to(device).eval()

def rebuild_pth(folder, out_path):
    with zipfile.ZipFile(out_path, "w", zipfile.ZIP_STORED) as z:
        for root, _, files in os.walk(folder):
            for fn in files:
                full = os.path.join(root, fn)
                z.write(full, "archive/" + os.path.relpath(full, folder))
    return out_path

def scan(folder, cands):
    for root, _, fs in os.walk(folder, followlinks=True):
        for fn in fs:
            p = os.path.join(root, fn)
            if fn.endswith(".pth"):
                cands.append(p)
            elif fn == "data.pkl":
                cands.append(rebuild_pth(root, f"/kaggle/working/rebuilt_{len(cands)}.pth"))
            elif fn.endswith(".zip") and os.path.getsize(p) < 2e9:
                with zipfile.ZipFile(p) as z:
                    names = z.namelist()
                    if any(n.endswith("data.pkl") for n in names) and not any(n.endswith(".pth") for n in names):
                        cands.append(p)
                    else:
                        dest = f"/kaggle/working/unzipped_{len(cands)}"
                        z.extractall(dest); scan(dest, cands)

model = load(NORMAL_PTH)
metric = DiceMetric(include_background=True, reduction="none")
_d = tf(test_files[0])
_x = torch.as_tensor(np.asarray(_d["image"]))[None].float().to(device); _x[:, 1] = 0
_y = torch.as_tensor(np.asarray(_d["label"]))[None].float().to(device)
def et_no_t1ce(m):
    with torch.no_grad(), torch.autocast("cuda"):
        o = sliding_window_inference(_x, (96, 96, 96), 4, m, overlap=0.5).float()
    return metric(y_pred=(torch.sigmoid(o) > 0.5).float(), y=_y)[0, 2].item()

cands = []; scan("/kaggle/input", cands)
print("Dropout candidates:", cands)
model_drop, best_et, drop_path = None, -1, None
for p in cands:
    try:
        m = load(p)
    except Exception:
        print("Skipped:", p); continue
    e = et_no_t1ce(m)
    print(f"ET without T1ce = {e:.2f}  ->  {p}")
    if e > best_et:
        model_drop, best_et, drop_path = m, e, p
    else:
        del m
    torch.cuda.empty_cache()
assert model_drop is not None, "No dropout model found. Add your old dataset as input."
print(f"Normal model : just trained (ET without T1ce = {et_no_t1ce(model):.2f})")
print(f"Dropout model: {drop_path} (ET without T1ce = {best_et:.2f})", flush=True)

# ===== 4. Helpers =====
KEEP = {"all4": [0, 1, 2, 3], "no_FLAIR": [1, 2, 3], "no_T1ce": [0, 2, 3], "no_T1": [0, 1, 3], "no_T2": [0, 1, 2]}
def drop(x, cond):
    xin = torch.zeros_like(x); xin[:, KEEP[cond]] = x[:, KEEP[cond]]
    return xin
def logits(m, x, overlap=0.5):
    with torch.no_grad(), torch.autocast("cuda"):
        return sliding_window_inference(x, (96, 96, 96), 4, m, overlap=overlap).float()
def dices(pred, y):
    return metric(y_pred=pred, y=y)[0].tolist()
def gradcam_window(m, x, ch):
    store = {}
    h1 = m.decoder2.register_forward_hook(lambda mod, i, o: store.__setitem__("a", o))
    h2 = m.decoder2.register_full_backward_hook(lambda mod, gi, go: store.__setitem__("g", go[0]))
    m.zero_grad()
    with torch.enable_grad():
        out = m(x)
        mask = (torch.sigmoid(out[:, ch]) > 0.5).float()
        score = (out[:, ch] * mask).sum() if mask.sum() > 0 else out[:, ch].sum()
        score.backward()
    h1.remove(); h2.remove()
    w = store["g"].mean(dim=(2, 3, 4), keepdim=True)
    cam = F.relu((w * store["a"]).sum(1, keepdim=True))
    return F.interpolate(cam, size=x.shape[2:], mode="trilinear", align_corners=False)[0, 0].detach().cpu().numpy()
def gradcam_tumor(m, vol, wt_mask, ch, S=96):
    full = np.zeros(vol.shape[1:], np.float32)
    if not wt_mask.any():
        return full
    lo = np.maximum(np.argwhere(wt_mask).min(0) - 8, 0)
    hi = np.argwhere(wt_mask).max(0) + 8
    starts = []
    for a, b, n in zip(lo, hi, vol.shape[1:]):
        s = list(range(int(a), int(max(a + 1, b - S + 1)), S)) + [int(max(a, b - S))]
        starts.append(sorted({max(0, min(v, n - S)) for v in s}))
    for x0 in starts[0]:
        for y0 in starts[1]:
            for z0 in starts[2]:
                x = torch.from_numpy(vol[:, x0:x0+S, y0:y0+S, z0:z0+S].copy())[None].float().to(device)
                region = full[x0:x0+S, y0:y0+S, z0:z0+S]
                np.maximum(region, gradcam_window(m, x, ch), out=region)
    return full / (full.max() + 1e-8)
loader = lambda: DataLoader(Dataset(data=test_files, transform=tf), batch_size=1, num_workers=2)

# ===== 5. Dice, 100 test patients =====
rows = []
for i, b in enumerate(loader()):
    x, y = b["image"].to(device), b["label"].float().to(device)
    for mname, m, conds in [("normal", model, ["all4", "no_FLAIR", "no_T1ce", "no_T1", "no_T2"]),
                            ("dropout", model_drop, ["all4", "no_FLAIR", "no_T1ce"])]:
        for c in conds:
            tc, wt, et = dices((torch.sigmoid(logits(m, drop(x, c))) > 0.5).float(), y)
            rows.append({"case": test_files[i]["id"], "model": mname, "condition": c, "TC": tc, "WT": wt, "ET": et})
    if (i + 1) % 10 == 0:
        print(f"Dice: {i+1}/100", flush=True)
dice_df = pd.DataFrame(rows)
dice_df.to_csv("/kaggle/working/dice_all_patients.csv", index=False)
print("\nTABLE 1: Mean Dice, 100 patients")
print(dice_df.groupby(["model", "condition"])[["TC", "WT", "ET"]].mean().round(3), flush=True)

# ===== 6. Uncertainty, 100 test patients =====
FLIPS = [[], [2], [3], [4], [2, 3], [2, 4], [3, 4], [2, 3, 4]]
urows = []
for i, b in enumerate(loader()):
    x, y = b["image"].to(device), b["label"].float().to(device)
    brain = x.abs().sum(1, keepdim=True) > 0
    for c in ["all4", "no_FLAIR", "no_T1ce"]:
        xin = drop(x, c); probs = []
        for dims in FLIPS:
            xi = torch.flip(xin, dims) if dims else xin
            o = logits(model, xi, overlap=0.25)
            probs.append(torch.sigmoid(torch.flip(o, dims) if dims else o))
        probs = torch.stack(probs)
        mean_p, std_p = probs.mean(0), probs.std(0)
        tc, wt, et = dices((mean_p > 0.5).float(), y)
        region = (mean_p > 0.05).any(1, keepdim=True).expand_as(std_p)
        urows.append({"case": test_files[i]["id"], "condition": c, "TC": tc, "WT": wt, "ET": et,
                      "unc_tumor": std_p[region].mean().item() if region.any() else 0.0,
                      "unc_brain": std_p[brain.expand_as(std_p)].mean().item()})
        del probs
    if (i + 1) % 10 == 0:
        print(f"Uncertainty: {i+1}/100", flush=True)
unc = pd.DataFrame(urows)
unc["mean_dice"] = unc[["TC", "WT", "ET"]].mean(axis=1)
unc.to_csv("/kaggle/working/uncertainty_all_patients.csv", index=False)
print("\nTABLE 2: Uncertainty, 100 patients")
print(unc.groupby("condition")[["mean_dice", "unc_tumor", "unc_brain"]].mean().round(4))
r, p = spearmanr(unc["unc_tumor"], unc["mean_dice"], nan_policy="omit")
print(f"Spearman: r = {r:.3f}, p = {p:.2e}", flush=True)

# ===== 7. Grad-CAM averages, 100 test patients =====
names, gc_rows = {0: "TC", 1: "WT", 2: "ET"}, []
for i, f in enumerate(test_files):
    d = tf(f)
    vol, lab = np.asarray(d["image"]), np.asarray(d["label"]).astype(bool)
    if lab[1].any():
        for mname, m in [("normal", model), ("dropout", model_drop)]:
            for c in ["all4", "no_T1ce"]:
                v = vol.copy()
                if c == "no_T1ce":
                    v[1] = 0
                for ch in [0, 1, 2]:
                    if lab[ch].any():
                        cam = gradcam_tumor(m, v, lab[1], ch)
                        gc_rows.append({"case": f["id"], "model": mname, "condition": c, "region": names[ch],
                                        "inside": cam[lab[ch]].sum() / (cam.sum() + 1e-8)})
    if (i + 1) % 10 == 0:
        print(f"Grad-CAM: {i+1}/100", flush=True)
gc = pd.DataFrame(gc_rows)
gc.to_csv("/kaggle/working/gradcam_all_patients.csv", index=False)
print("\nTABLE 3: Attention inside the real region (%)")
print((gc.pivot_table(index=["model", "condition"], columns="region", values="inside") * 100).round(0), flush=True)

# ===== 8. 20 website patients =====
WEB = Path("/kaggle/working/website_patients"); WEB.mkdir(exist_ok=True)
n_all = dice_df[(dice_df.model == "normal") & (dice_df.condition == "all4")].copy()
n_all["mean"] = n_all[["TC", "WT", "ET"]].mean(axis=1, skipna=True)
n_all = n_all.sort_values("mean", ascending=False).reset_index(drop=True)
picks = n_all.iloc[np.linspace(0, len(n_all) - 1, 20).round().astype(int)]
by_id = {f["id"]: f for f in test_files}

def to_labels(pred):
    out = np.zeros(pred.shape[1:], np.uint8)
    out[pred[1]] = 2; out[pred[0]] = 1; out[pred[2]] = 4
    return out

def location(mask, brain, affine):
    codes = nib.aff2axcodes(affine)
    c, mid = np.argwhere(mask).mean(0), np.argwhere(brain).mean(0)
    words = {"L": ("left", "right"), "R": ("right", "left"), "A": ("front", "back"),
             "P": ("back", "front"), "S": ("upper", "lower"), "I": ("lower", "upper")}
    return ", ".join(words[code][0] if c[k] > mid[k] else words[code][1] for k, code in enumerate(codes))

report = []
for rank, (_, row) in enumerate(picks.iterrows()):
    f = by_id[row["case"]]
    d = tf(f)
    vol, lab = np.asarray(d["image"]), np.asarray(d["label"]).astype(bool)
    x = torch.from_numpy(vol)[None].float().to(device)
    pred = (torch.sigmoid(logits(model, x)) > 0.5)[0].cpu().numpy()
    pred_no = (torch.sigmoid(logits(model, drop(x, "no_T1ce"))) > 0.5)[0].cpu().numpy()
    img = nib.load(f["image"][0]); aff, hdr = img.affine, img.header
    pdir = WEB / f["id"]; pdir.mkdir(exist_ok=True)
    shutil.copy(f["image"][0], pdir / "flair.nii.gz")
    shutil.copy(f["label"], pdir / "truth.nii.gz")
    nib.save(nib.Nifti1Image(to_labels(pred), aff), pdir / "prediction.nii.gz")
    nib.save(nib.Nifti1Image(to_labels(pred_no), aff), pdir / "prediction_no_t1ce.nii.gz")
    for name, ch in [("wt", 1), ("tc", 0), ("et", 2)]:
        cam = gradcam_tumor(model, vol, pred[1] if pred[1].any() else lab[1], ch)
        nib.save(nib.Nifti1Image((cam * 255).astype(np.uint8), aff), pdir / f"gradcam_{name}.nii.gz")
    vox = float(np.prod(hdr.get_zooms()[:3])) / 1000
    no_row = dice_df[(dice_df.case == f["id"]) & (dice_df.model == "normal") & (dice_df.condition == "no_T1ce")].iloc[0]
    kind = "best" if rank < 4 else ("hard" if rank >= 16 else "typical")
    report.append({
        "id": f["id"], "case_type": kind, "tumor_type": "Glioma (all BraTS 2021 patients)",
        "dice": {k: round(row[k], 3) for k in ["WT", "TC", "ET"]},
        "dice_without_t1ce": {k: round(no_row[k], 3) for k in ["WT", "TC", "ET"]},
        "volume_cm3": {"WT": round(pred[1].sum() * vox, 1), "TC": round(pred[0].sum() * vox, 1), "ET": round(pred[2].sum() * vox, 1)},
        "location_approx": location(lab[1] if lab[1].any() else pred[1], vol[0] != 0, aff),
    })
    print(f"Website patient {rank+1}/20: {f['id']} ({kind})", flush=True)
json.dump(report, open(WEB / "patients.json", "w"), indent=1)

# ===== 9. Pack everything =====
shutil.rmtree(OUT, ignore_errors=True)
!cd /kaggle/working && zip -qr final_results.zip normal_model.pth history.json dice_all_patients.csv uncertainty_all_patients.csv gradcam_all_patients.csv website_patients && ls -lh final_results.zip
print(f"ALL DONE in {(time.time()-START)/3600:.1f} hours: download final_results.zip from Output")
   

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 52.7 MB/s eta 0:00:00
199 train | 49 val | 100 test


nvImageCodec is not available; skipping pydicom decoder plugin registration.
/usr/local/lib/python3.13/dist-packages/monai/data/image_reader.py:1166: UserWarning: NvImgCodecPydicomReader: nvImageCodec decoder plugin did not register successfully. Falling back to default pydicom decoders.
  warnings.warn(
nvImageCodec is not available; skipping pydicom decoder plugin registration.
/tmp/ipykernel_24/2196912117.py:75: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Epoch 1/100  loss 0.8069  (2.0 min)
Epoch 2/100  loss 0.6999  (2.0 min) | val Dice 0.6064 (TC 0.579 WT 0.823 ET 0.417)  -> saved new best model
Epoch 3/100  loss 0.6394  (2.0 min)
Epoch 4/100  loss 0.5533  (2.0 min) | val Dice 0.7995 (TC 0.783 WT 0.867 ET 0.748)  -> saved new best model
Epoch 5/100  loss 0.4906  (2.0 min)
Epoch 6/100  loss 0.4018  (2.0 min) | val Dice 0.7816 (TC 0.749 WT 0.838 ET 0.758)
Epoch 7/100  loss 0.3970  (2.0 min)
Epoch 8/100  loss 0.3973  (2.0 min) | val Dice 0.8353 (TC 0.795 WT 0.901 ET 0.810)  -> saved new best model
Epoch 9/100  loss 0.3834  (2.0 min)
Epoch 10/100  loss 0.3924  (2.0 min) | val Dice 0.8342 (TC 0.792 WT 0.894 ET 0.817)
Epoch 11/100  loss 0.3329  (2.0 min)
Epoch 12/100  loss 0.3508  (2.0 min) | val Dice 0.8165 (TC 0.773 WT 0.890 ET 0.787)
Epoch 13/100  loss 0.3621  (2.0 min)
Epoch 14/100  loss 0.3732  (2.0 min) | val Dice 0.7940 (TC 0.723 WT 0.867 ET 0.791)
Epoch 15/100  loss 0.3919  (2.0 min)
Epoch 16/100  loss 0.4079  (2.0 min) | val Dice 0.

OutOfMemoryError: CUDA out of memory. Tried to allocate 162.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 52.81 MiB is free. Including non-PyTorch memory, this process has 14.51 GiB memory in use. Of the allocated memory 13.08 GiB is allocated by PyTorch, and 1.24 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)